# ADK learning lab (local venv)

Same learning arc as `workshop/ADK_Learning_tools.ipynb` **without Colab**: use a virtual environment on your laptop.

**Before you start:**

```bash
cd workshop
python3 -m venv .venv
source .venv/bin/activate
pip install -U pip
pip install -r requirements-workshop.txt
export GOOGLE_API_KEY="your-key"
```

Optional Jupyter kernel: `python -m ipykernel install --user --name=adk-workshop` then select it in this notebook.

## Part 0: Imports and configuration

Uses the **Gemini API** via `GOOGLE_API_KEY`. For Vertex AI instead, follow [ADK authentication](https://google.github.io/adk-docs/).

In [ ]:
import asyncio
import os

from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.adk.tools.tool_context import ToolContext
from google.genai.types import Content, Part
from IPython.display import Markdown, display

if not os.environ.get("GOOGLE_API_KEY"):
    print("Warning: GOOGLE_API_KEY is not set. Set it before running agent cells.")
else:
    print("OK: GOOGLE_API_KEY is set.")

## Part 1: First agent — Day trip with Google Search

Defines `day_trip_agent` and runs one query through `Runner` + `InMemorySessionService`.

In [ ]:
def create_day_trip_agent() -> Agent:
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash-lite",
        description="Full-day itineraries with mood, interests, and budget.",
        instruction="""
        You create same-day itineraries. Respect budget (frugal / mid / splurge).
        Structure: morning, afternoon, evening. Use Google Search when it helps
        with venues or hours. Return Markdown with clear headings.
        """,
        tools=[google_search],
    )


session_service = InMemorySessionService()
user_id = "workshop_user_001"
day_trip_agent = create_day_trip_agent()
print(f"Agent ready: {day_trip_agent.name}")

In [ ]:
def _text_from_event(event) -> str | None:
    if not event.content or not event.content.parts:
        return None
    for part in event.content.parts:
        if part.text:
            return part.text
    return None


async def run_agent_query(agent: Agent, query: str, session_id: str) -> str:
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name,
    )
    final_text = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=Content(parts=[Part(text=query)], role="user"),
    ):
        t = _text_from_event(event)
        if t and event.is_final_response():
            final_text = t
    return final_text


async def demo_day_trip():
    session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=user_id,
    )
    q = "Plan a relaxed artsy day in Barcelona on a modest budget. Markdown itinerary."
    print("Query:", q)
    reply = await run_agent_query(day_trip_agent, q, session.id)
    display(Markdown(reply or "(no text response)"))


await demo_day_trip()

## Part 2: Custom function tools

Type hints and docstrings define tool schemas for the model (see also `workshop/demos/custom_tools`).

In [ ]:
import datetime
from zoneinfo import ZoneInfo


def get_weather(city: str) -> dict:
    """Return a stub weather report (demo)."""
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": "Sunny, 25 °C in New York.",
        }
    return {"status": "error", "error_message": f"No data for {city}."}


def get_current_time(city: str) -> dict:
    """Return local time for supported cities (demo)."""
    if city.lower() != "new york":
        return {"status": "error", "error_message": f"Unknown city: {city}"}
    now = datetime.datetime.now(ZoneInfo("America/New_York"))
    return {
        "status": "success",
        "report": now.strftime("%Y-%m-%d %H:%M:%S %Z"),
    }


weather_agent = Agent(
    name="weather_workshop_agent",
    model="gemini-2.5-flash-lite",
    instruction="Use tools to answer. Explain tool errors clearly.",
    tools=[get_weather, get_current_time],
)
print(f"Agent ready: {weather_agent.name}")

In [ ]:
async def demo_weather_tools():
    session = await session_service.create_session(
        app_name=weather_agent.name,
        user_id=user_id,
    )
    q = "What is the weather and local time in New York?"
    print("Query:", q)
    reply = await run_agent_query(weather_agent, q, session.id)
    display(Markdown(reply or "(no text response)"))


await demo_weather_tools()

## Part 3: Session memory (`ToolContext.state`)

State written in a tool is visible on later turns **in the same session** (see `workshop/demos/session_memory`).

In [ ]:
def remember_display_name(display_name: str, tool_context: ToolContext) -> str:
    """Store the user's display name for this session."""
    tool_context.state["user_display_name"] = display_name.strip()
    return f"Remembered: {display_name.strip()}"


def recall_display_name(tool_context: ToolContext) -> str:
    """Read the stored display name."""
    name = tool_context.state.get("user_display_name")
    if not name:
        return "No name stored yet."
    return str(name)


memory_agent = Agent(
    name="memory_workshop_agent",
    model="gemini-2.5-flash-lite",
    instruction=(
        "When the user gives a name, call remember_display_name. "
        "When they ask what you remember, call recall_display_name."
    ),
    tools=[remember_display_name, recall_display_name],
)
print(f"Agent ready: {memory_agent.name}")

In [ ]:
async def demo_memory_two_turns():
    session = await session_service.create_session(
        app_name=memory_agent.name,
        user_id=user_id,
    )
    sid = session.id
    print("--- Turn 1 ---")
    r1 = await run_agent_query(
        memory_agent,
        "Please remember my display name is Morgan.",
        sid,
    )
    display(Markdown(r1 or "(no reply)"))
    print("--- Turn 2 ---")
    r2 = await run_agent_query(memory_agent, "What is my display name?", sid)
    display(Markdown(r2 or "(no reply)"))
    print("If you see Morgan in turn 2, session state is working.")


await demo_memory_two_turns()
